In [42]:
import pandas as pd
import numpy as np
import pyvinecopulib as pv

from pandas_datareader import data as pdr
from scipy.stats import t

# Download data

In [71]:
# Nasdaq Composite
# ndq = pdr.DataReader("GC.F", "stooq", start, end)
# ndq#["Close"]

In [ ]:
tickers = ["^DAX", "^SPX", "^NDQ", "10YDEY.B", "10YUSY.B"]  # stooq uses lowercase for US "PLOPLN3M"
start = "2016-01-01"
end = "2026-01-01"

dfs = {}
for t in tickers:
    df = pdr.DataReader(t, "stooq", start, end)   # returns newest->oldest
    print(t)
    df = df.sort_index()
    dfs[t.upper()] = df["Close"].rename(t.upper())

closes = pd.concat(dfs.values(), axis=1).dropna()
print(closes.tail())

^DAX
^SPX
^NDQ
10YDEY.B
                ^DAX     ^SPX      ^NDQ  10YDEY.B
Date                                             
2025-12-19  24288.40  6834.50  23307.62    2.8950
2025-12-22  24283.97  6878.49  23428.83    2.9001
2025-12-23  24340.06  6909.79  23561.84    2.8624
2025-12-29  24351.12  6905.74  23474.35    2.8298
2025-12-30  24490.41  6896.24  23419.08    2.8546


# Prepare data

In [4]:
def t_winsorize(series, alpha = 0.005):
    x = series.dropna()
    df_t, loc_t, scale_t = t.fit(x)

    lower_bound = t.ppf(alpha, df_t, loc_t, scale_t)
    upper_bound = t.ppf(1 - alpha, df_t, loc_t, scale_t)

    # Do not modify assets with no extremes
    if x.min() >= lower_bound and x.max() <= upper_bound:
        return series.copy()
    
    series_new = series.copy()
    series_new = series_new.clip(lower_bound, upper_bound)

    return series_new

In [16]:
def to_log_returns(df):
    return np.log(df).diff().dropna()  

def recalibrate_returns(dicito, mu, sigma, print_names: bool = False):
    dic = dicito.copy()
    for key in dic.keys():
        dic[key] = dic[key] * sigma[key] + mu[key]
        if print_names:
            print(key)
            
    return dic

In [ ]:
# read data
# Fallback data, we will eventually use the indices data when we gather all of it
data = pd.read_excel("10_asset_prices.xlsx").set_index("Date")
log_data = to_log_returns(data)
# If we only want monthly data we can resample
# monthly = (1 + data).resample("M").prod() - 1

# standardize data
data_standard = (log_data - log_data.mean()) / log_data.std()

In [20]:
df = data_standard.copy()
df = df.apply(lambda col: t_winsorize(col))

U = pv.to_pseudo_obs(df.to_numpy())

controls = pv.FitControlsVinecop(
    family_set = [
        pv.student, pv.gaussian,
        pv.clayton, pv.gumbel
        # pv.frank, pv.joe
    ],
    # trunc_lvl = 3,
    select_trunc_lvl = True,
    show_trace = False
)

cop = pv.Vinecop.from_data(U, controls = controls)

In [26]:
number_of_assets = data_standard.shape[1]
N = 252
sim = 1000
n_sim = N * sim

In [27]:
u_sim = cop.simulate(n_sim)

In [28]:
x_sim = np.asarray([np.quantile(df.to_numpy()[:, i], u_sim[:, i]) for i in range(0, number_of_assets)])
Usim_vine = x_sim.reshape(sim, N, number_of_assets).transpose(1,0,2)

cols = df.columns

simulated_dict_vine = {
    col: pd.DataFrame(index = range(N),
        columns = [f"sim_{i}" for i in range(sim)],
        dtype = float)
    for col in cols
}

for j, col in enumerate(cols):
    U_asset = Usim_vine[:,:,j]
    simulated_dict_vine[col].loc[:,:] = U_asset

In [ ]:
# This is wrong but just a placeholder for now, we recalibrate with mean and vol
recalibrated_sim = recalibrate_returns(simulated_dict_vine, log_data.mean(), log_data.std())

Eventually to come, graphing the paths, calculating VaR and maxium drawdowns for each asset